
### Middleware

In [4]:
import os
from  dotenv import load_dotenv
load_dotenv() 


os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [3]:
# summarization middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# message based summerization middleware
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("messages",10),
        keep=("messages",4)
    )],
)

In [ ]:
# Run with thread id

config = {
    "configurable": {
        "thread_id": "test-1"
    }
    
    
}

In [7]:
questions = [
    "what is 3 + 3?",
    "what is 4 + 4?",
    "what is 5 + 5?",
    "what is 6 + 6?",
]

for q in questions:
    res = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(res)
    print(len(res["messages"]))
    print("--------------------------------------------------")

{'messages': [HumanMessage(content='what is 3 + 3?', additional_kwargs={}, response_metadata={}, id='ad489fd7-7179-41c8-b489-9c31706a5c65'), AIMessage(content='3 + 3 = 6', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01b90-c980-7ba0-ac41-2f3d1ce29045-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 33, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 26}})]}
2
--------------------------------------------------
{'messages': [HumanMessage(content='what is 3 + 3?', additional_kwargs={}, response_metadata={}, id='ad489fd7-7179-41c8-b489-9c31706a5c65'), AIMessage(content='3 + 3 = 6', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a01b90-c980-7ba0-ac4

In [11]:
# summarization middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool


@tool("setup_search",description="setup search tool")
def search(query: str) -> str:
    return f"search results for {query}"

# message based summerization middleware
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("tokens",550),
        keep=("tokens",200)
    )],
)
config = {
    "configurable": {
        "thread_id": "test-2"
    }
}


def count_tokens(messages):
    return sum(len(m.content.split()) for m in messages)


### Human In the Loop Middleware

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
@tool("read_email_tool",description="read email tool")
def read_email_tool(email_id) -> str:
    """read email tool"""
    return f"Email content for ID {email_id}"

@tool("send_email_tool",description="send_email_tool")
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """send email tool"""
    return f"Email sent to {recipient} with subject {subject}"


In [9]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [23]:

config = {
    "configurable": {
        "thread_id": "test-approve"
    }
}

result = agent.invoke(
    {"messages": [
        HumanMessage(content="Send email to kishanghosh090@gmail.com as HI, sub hello")
    ]},
    config=config
)

In [17]:
result

{'messages': [HumanMessage(content='Send email to kishanghosh090@gmail.com as HI, sub hello', additional_kwargs={}, response_metadata={}, id='2113da46-0101-4691-90cb-940ce9159625'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "hello", "body": "HI", "recipient": "kishanghosh090@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'376afe0d-1a9f-4db5-8777-cc1c3d7e8cdb': 'Co0CARFNMg+0rwwXsLiFS1R1V3+K6Y4TPjen8WW3RhxGkgjts0EDcpIZrovHFCrtzari7PSDPlWalhs8zcMBvLsVTYPPBvgX26Ycy59/PSr8cGzIxgQBwFHhfsCimnfxfWLmVzySRxXXGFXEH+ER6j8Ce/VJE0q6YQoEVxgIkdmtyblVdvjPbRrRlnEI6XLSvn6BeNhaNw92pAJjPmJFom4YYyWPR88h+jRXrAP2jA6BbatBsRyJEs0id4Qp5lkODR5NggtCbbz75cs3+ykhgaFlac9i5FNleQp0UqcQvxEtF2toyIbEGhKSiQGm86IxyUoLdc7OJPQEzN3o/QQ8fCRv/iVijZhKWJMeHdtDYC0='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a027b9-fbaf-7572-b360-6021ec36

In [24]:
from langgraph_sdk.schema import Command


if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    ),

    print(result[0]['messages'][-1].content)

Paused! Approving...

